# **Decision Tree Classification and Regression**

Completed using Iris dataset (Classification) and Diabetes dataset (Regression)

# Classification
## Load the data

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
X_class = iris.data
y_class = iris.target

## Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

## Modeling
### Decision Tree - Without specifing hyperparameters

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import time

dt_default = DecisionTreeClassifier(random_state=42)

start_time = time.time()
dt_default.fit(X_train_c_scaled, y_train_c)
time_dt_default = time.time() - start_time

print(f"Training Time (Default): {time_dt_default:.6f} seconds")

Training Time (Default): 0.005010 seconds


### Decision Tree - Add Hyperparameters

In [ ]:
dt_custom = DecisionTreeClassifier(max_depth=3, min_samples_split=4, random_state=42)

start_time = time.time()
dt_custom.fit(X_train_c_scaled, y_train_c)
time_dt_custom = time.time() - start_time

print(f"Training Time (Custom): {time_dt_custom:.6f} seconds")

Training Time (Custom): 0.003114 seconds


### Decision Tree - Hyperparameters tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 4, 5, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)

start_time = time.time()
grid_search.fit(X_train_c_scaled, y_train_c)
time_dt_grid = time.time() - start_time

best_dt_model = grid_search.best_estimator_
print(f"Training Time (GridSearch): {time_dt_grid:.6f} seconds")
print(f"Best Parameters: {grid_search.best_params_}")

Training Time (GridSearch): 0.316900 seconds
Best Parameters: {'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 2}


### Best model metrics

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_pred_c = best_dt_model.predict(X_test_c_scaled)
acc_c = accuracy_score(y_test_c, y_pred_c)
f1_c = f1_score(y_test_c, y_pred_c, average='weighted')

print(f"Best Decision Tree Accuracy: {acc_c:.4f}")
print(f"Best Decision Tree F1 Score: {f1_c:.4f}")

Best Decision Tree Accuracy: 1.0000
Best Decision Tree F1 Score: 1.0000


# Regression
## Load the data

In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X_reg = diabetes.data
y_reg = diabetes.target

## Data Preprocessing

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

## Modeling
### Model 1: Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

dt_reg = DecisionTreeRegressor(max_depth=4, random_state=42)

start_time = time.time()
dt_reg.fit(X_train_r_scaled, y_train_r)
time_dt_reg = time.time() - start_time

mse_dt = mean_squared_error(y_test_r, dt_reg.predict(X_test_r_scaled))
print(f"DT Regressor Training Time: {time_dt_reg:.6f} seconds | MSE: {mse_dt:.2f}")

DT Regressor Training Time: 0.002679 seconds | MSE: 3568.97


### Model 2: Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lr_reg = LinearRegression()

start_time = time.time()
lr_reg.fit(X_train_r_scaled, y_train_r)
time_lr_reg = time.time() - start_time

mse_lr = mean_squared_error(y_test_r, lr_reg.predict(X_test_r_scaled))
print(f"Linear Regression Training Time: {time_lr_reg:.6f} seconds | MSE: {mse_lr:.2f}")

Linear Regression Training Time: 0.023567 seconds | MSE: 2900.19


### Model 3: Support Vector Regressor (SVR)

In [ ]:
from sklearn.svm import SVR

svr_reg = SVR(kernel='linear')

start_time = time.time()
svr_reg.fit(X_train_r_scaled, y_train_r)
time_svr_reg = time.time() - start_time

mse_svr = mean_squared_error(y_test_r, svr_reg.predict(X_test_r_scaled))
print(f"SVR Training Time: {time_svr_reg:.6f} seconds | MSE: {mse_svr:.2f}")

SVR Training Time: 0.010695 seconds | MSE: 2939.81


# Results

## 1. Classification Comparison (Iris Dataset)

### Results Summary
| Model Configuration | Training Time (Sec) | Accuracy | F1 Score |
| :--- | :--- | :--- | :--- |
| Decision Tree (Default) | ~0.0050 | 1.0000 | 1.0000 |
| Decision Tree (Custom Hyperparameters) | ~0.0031 | 1.0000 | 1.0000 |
| Decision Tree (GridSearch Tuning) | ~0.3169 | 1.0000 | 1.0000 |

### Justification of Results
* **Training Time:** The standard and custom Decision Trees trained extremely fast because the CART algorithm relies on a straightforward, greedy approach to calculate splits. The custom model (`max_depth=3`) was slightly faster than the default because restricting the depth reduces the number of splits. The GridSearch model took significantly longer because it iterates through multiple combinations of hyperparameters using 5-fold cross-validation.
* **Performance:** All three configurations achieved a perfect 1.0 accuracy and F1 score. This occurs because the Iris dataset features cleanly separable classes that are easily partitioned by orthogonal splits.
* **Best Model:** The Decision Tree tuned via GridSearch is the most robust model. The cross-validation process ensures that this perfect accuracy is generalized and not merely a result of a lucky train/test split.

---

## 2. Regression Comparison (Diabetes Dataset)

### Results Summary
| Regression Model | Training Time (Sec) | Mean Squared Error (MSE) |
| :--- | :--- | :--- |
| Decision Tree Regressor (`max_depth=4`) | ~0.0026 | 3568.97 |
| Linear Regression | ~0.0235 | 2900.19 |
| Support Vector Regressor (Linear) | ~0.0106 | 2939.81 |

### Justification of Results
* **Training Time:** The Decision Tree Regressor was the fastest to train, as it simply calculates variance reductions for binary splits. SVR and Linear Regression took slightly longer due to the mathematical optimization required (solving for coefficients and support vectors).
* **Performance (MSE):** The Decision Tree performed the worst, yielding the highest MSE. Decision Trees predict the average value of a leaf node, creating a "step-function" boundary which struggles to smoothly model continuous linear trends. Both Linear Regression and the Linear SVR performed significantly better, indicating that the relationships in the Diabetes dataset are predominantly linear.
* **Best Model:** The Linear Regression model is the best performing model for this dataset, achieving the lowest MSE (2900.19) by successfully capturing the linear correlations.

---

## 3. Comparison with SVM

### Performance Comparison
* Using the Iris dataset, the Support Vector Machine with a Polynomial Kernel (Degree 3) achieved excellent accuracy, matching the 1.0000 accuracy of the Decision Trees.
* While both algorithms handle the simple Iris data well, SVM uses the "kernel trick" to map data into a higher dimension to draw a complex decision boundary, whereas the Decision Tree creates a set of simple, perpendicular if/else rules.

### Training Time Justification
* Generally, a standard Decision Tree trains faster than an SVM.
* The CART algorithm's greedy, feature-by-feature splitting is computationally lighter compared to an SVM, which requires solving a complex quadratic programming problem to maximize the margin between support vectors.
* However, when hyperparameter tuning (GridSearch) is introduced to the Decision Tree, it becomes much slower than a single run of an SVM.